# Method A sandbox
**Purpose.** Test method A: kernel anchored at $t$ with $r_t = 0$ — against same-day fit and `pred.shift(1)`

**Inputs.** Yahoo Finance `^GSPC` and `^VIX` 1995-01-01 → 2022-05-15.

**Expected runtime.** ~2 min on M5 Pro CPU (Guyon TSPL fit dominates).

**Dependencies.** `code_section6.motivation`, `code_section6.sandbox.method_a.compute`, `code_guyon.empirical_study.main_function`.

## Imports

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

import code_section6.motivation as motiv
from code_section6.sandbox.method_a.compute import method_a_forecast, evaluate_forecast
from empirical_study.main_function import perform_empirical_study

## Fetch SPX, VIX

In [ ]:
spx, vix = motiv.fetch_spx_vix_yahoo('1995-01-01', '2022-05-15')
spx_returns = spx.pct_change().dropna()
print('spx rows:', len(spx), '  vix rows:', len(vix), '  spx_returns rows:', len(spx_returns))

## Fit Guyon model

In [ ]:
sol = perform_empirical_study(
    vol=vix, index=spx, p=1, tspl=True,
    setting=[(1, 1), (2, 1/2)],
    train_start_date=pd.Timestamp('2000-01-01'),
    test_start_date=pd.Timestamp('2019-01-01'),
    test_end_date=pd.Timestamp('2022-05-15'),
    max_delta=1000,
)
print({k: sol[k] for k in ['train_r2', 'test_r2', 'train_rmse', 'test_rmse']})

## Extract fitted parameters

In [ ]:
p = sol['opt_params']
params = {
    'beta_0': float(p['beta_0']),
    'beta_1': float(p['beta_1']),
    'beta_2': float(p['beta_2']),
    'alpha_1': float(p['alpha_1']),
    'delta_1': float(p['delta_1']),
    'alpha_2': float(p['alpha_2']),
    'delta_2': float(p['delta_2']),
}
params

## Compute method A forecast on test set

In [ ]:
test_dates = sol['test_pred'].index
method_a = method_a_forecast(spx_returns, test_dates, params)
print('method_a head:')
print(method_a.head())

## Build forecast collection

In [ ]:
vix_test = vix.reindex(test_dates).dropna()
forecasts = {
    'same_day_fit': sol['test_pred'].reindex(vix_test.index),
    'method_a':     method_a.reindex(vix_test.index),
    'pred_shift_1': sol['test_pred'].shift(1).reindex(vix_test.index),
    'persistence':  vix.shift(1).reindex(vix_test.index),
}
for name, f in forecasts.items():
    print(f'{name:18s}  n={f.dropna().shape[0]:4d}')

## Focused plot: VIX, same-day fit, Method A.


In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(vix_test.index, 100 * vix_test,                  label='VIX (true)',         lw=1.0, color='black')
ax.plot(vix_test.index, 100 * forecasts['same_day_fit'], label='Same-day fit',       lw=1.2, color='C0')
ax.plot(vix_test.index, 100 * forecasts['method_a'],     label='Method A',           lw=1.2, color='C2')
ax.set_ylabel('VIX (%)')
ax.set_title('Same-day fit vs Method A — 2019-01-02 to 2022-05-13')
ax.legend()
fig.tight_layout()
out_path = OUT_DIR / 'fit_vs_method_a_full.pdf'
fig.savefig(out_path)
print('saved:', out_path)
plt.show()

## Zoom: 2020 COVID window

In [ ]:
zoom = (vix_test.index >= '2020-02-01') & (vix_test.index <= '2020-05-31')
fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(vix_test.index[zoom], 100 * vix_test[zoom],                  label='VIX (true)',   lw=1.0, color='black')
ax.plot(vix_test.index[zoom], 100 * forecasts['same_day_fit'][zoom], label='Same-day fit', lw=1.2, color='C0')
ax.plot(vix_test.index[zoom], 100 * forecasts['method_a'][zoom],     label='Method A',     lw=1.2, color='C2')
ax.set_ylabel('VIX (%)')
ax.set_title('Zoom: 2020-02 to 2020-05 (COVID)')
ax.legend()
fig.tight_layout()
out_path = OUT_DIR / 'fit_vs_method_a_covid.pdf'
fig.savefig(out_path)
print('saved:', out_path)
plt.show()

## Dynamics + level metrics table (all four candidates)

In [ ]:
rows = {name: evaluate_forecast(f, vix_test) for name, f in forecasts.items()}
metrics_df = pd.DataFrame(rows).T
metrics_df = metrics_df[['n_obs', 'rmse', 'sign_hit_rate', 'sign_hit_calm', 'sign_hit_volatile', 'directional_corr', 'mz_alpha', 'mz_beta']]
print(metrics_df.to_string(formatters={
    'rmse': '{:.4f}'.format,
    'sign_hit_rate': '{:.3f}'.format,
    'sign_hit_calm': '{:.3f}'.format,
    'sign_hit_volatile': '{:.3f}'.format,
    'directional_corr': '{:.3f}'.format,
    'mz_alpha': '{:.4f}'.format,
    'mz_beta': '{:.3f}'.format,
    'n_obs': '{:.0f}'.format,
}))

## One-line summary

In [ ]:
for name in ['same_day_fit', 'method_a', 'pred_shift_1', 'persistence']:
    r = rows[name]
    print(f"{name:14s}  RMSE={100*r['rmse']:.3f}%   sign-hit={r['sign_hit_rate']:.3f}   dir-corr={r['directional_corr']:+.3f}")